In [1]:
import copy
import cv2
import json
import open3d as o3d
import os
import numpy as np

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Constants
ARUCO_DICT_TYPE = cv2.aruco.DICT_5X5_50
MARKER_LENGTH = 0.05
CAMERA_MATRIX = np.array(
    [[3800, 0, 1440], [0, 3800, 1080], [0, 0, 1]], dtype=np.float32
)

DIST_COEFFS = np.zeros((5, 1), dtype=np.float32)
WINDOW_NAME = "ArUco Marker"
WINDOW_WIDTH = 800
WINDOW_HEIGHT = 500
AXIS_LENGTH = 0.1
SUPPORTED_EXTENSIONS = [".jpg", ".jpeg", ".png"]

In [14]:
def process_single_image(sfm_file_path, pcd_path):
    """
    Process a single image to detect ArUco markers and visualize them on the point cloud.
    
    Args:
        sfm_file_path (str): Path to the SfM data file
        pcd_path (str): Path to the point cloud file
    """
    # Step 1: Load SfM data to get camera pose
    with open(sfm_file_path, "r") as f:
        sfm_data = json.load(f)
    
    # Find the view corresponding to our image
    view_info = None
    
    for view in sfm_data["views"]:
        view_info = view
        break
    
    # Get camera pose
    image_filename = view_info["path"]
    poseId = view_info["poseId"]
    pose_info = next(
        (p["pose"]["transform"] for p in sfm_data["poses"] if p["poseId"] == poseId),
        None
    )
    
    if not pose_info:
        print(f"Error: No pose information found for image {image_filename}")
        return
    
    # Get camera rotation and center
    camera_rotation = np.array(pose_info["rotation"], dtype=float).reshape(3, 3)
    camera_center = np.array(pose_info["center"], dtype=float)
    
    # Step 2: Detect ArUco markers in the image
    img_path = view_info["path"]
    img = cv2.imread(img_path)
    if img is None:
        print(f"Error: Could not read image at {img_path}")
        return
    
    aruco_dict = cv2.aruco.getPredefinedDictionary(ARUCO_DICT_TYPE)
    parameters = cv2.aruco.DetectorParameters()
    
    corners, ids, _ = cv2.aruco.detectMarkers(img, aruco_dict, parameters=parameters)
    
    if ids is None:
        print(f"No markers detected in {image_filename}")
        return
    
    rvecs, tvecs, _ = cv2.aruco.estimatePoseSingleMarkers(
        corners, MARKER_LENGTH, CAMERA_MATRIX, DIST_COEFFS
    )
    
    # Draw detected markers on the image for verification
    img_with_marker = img.copy()
    img_with_marker = cv2.aruco.drawDetectedMarkers(img_with_marker, corners, ids)
    for i in range(len(ids)):
        cv2.drawFrameAxes(
            img_with_marker, CAMERA_MATRIX, DIST_COEFFS, rvecs[i], tvecs[i], AXIS_LENGTH
        )
    
    # Step 3: Convert marker positions to global coordinates
    marker_data = []
    
    for i in range(len(ids)):
        marker_id = ids[i][0]
        rvec = rvecs[i]
        tvec = tvecs[i]
        
        # Convert rvec to rotation matrix
        R_marker, _ = cv2.Rodrigues(rvec)
        
        # Transformation from marker to camera
        T_marker_to_cam = np.eye(4)
        T_marker_to_cam[:3, 3] = tvec.flatten()
        T_marker_to_cam[:3, :3] = R_marker
        
        # Transformation from camera to global
        T_cam_to_global = np.eye(4)
        T_cam_to_global[:3, 3] = camera_center
        T_cam_to_global[:3, :3] = camera_rotation
        
        # Marker to global
        T_marker_to_global = T_cam_to_global @ T_marker_to_cam
        
        position = T_marker_to_global[:3, 3]
        rotation = T_marker_to_global[:3, :3]
        
        marker_data.append({
            "marker_id": marker_id,
            "global_position": position.tolist(),
            "global_rotation": rotation.tolist()
        })
        
        print(f"Marker {marker_id} global position: {position}")
    
    # Step 4: Load and visualize the point cloud with marker and camera positions
    pcd = o3d.io.read_point_cloud(pcd_path)
    print(f"Point cloud loaded with {len(pcd.points)} points")
    
    # Create a list of geometries to display
    geometries = [pcd]
    
    # Add camera position and orientation
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
    camera_frame.translate(camera_center)
    camera_frame.rotate(camera_rotation)
    
    # Add a sphere to highlight the camera position
    camera_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.02)
    camera_sphere.paint_uniform_color([0, 1, 1])  # Cyan
    camera_sphere.translate(camera_center)
    
    geometries.append(camera_frame)
    geometries.append(camera_sphere)
    
    # Add marker positions and orientations
    for marker in marker_data:
        # Create coordinate frame for marker
        marker_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.05)
        marker_frame.translate(marker["global_position"])
        marker_frame.rotate(np.array(marker["global_rotation"]))
        
        # Add a sphere to highlight the marker position
        marker_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.01)
        marker_sphere.paint_uniform_color([1, 0, 1])  # Magenta
        marker_sphere.translate(marker["global_position"])
        
        geometries.append(marker_frame)
        geometries.append(marker_sphere)
        
        print(f"Added marker {marker['marker_id']} to visualization")
    
    # Draw a line connecting camera to each marker
    for marker in marker_data:
        marker_pos = np.array(marker["global_position"])
        
        line_points = [camera_center, marker_pos]
        line_indices = [[0, 1]]
        line_colors = [[1, 1, 0]]  # Yellow
        
        line = o3d.geometry.LineSet()
        line.points = o3d.utility.Vector3dVector(line_points)
        line.lines = o3d.utility.Vector2iVector(line_indices)
        line.colors = o3d.utility.Vector3dVector(line_colors)
        
        geometries.append(line)
    
    # Visualize everything
    print("Displaying 3D visualization...")
    o3d.visualization.draw_geometries(
        geometries,
        window_name="Marker and Camera Positions",
        width=1200,
        height=800
    )
        
    # Display the result
    cv2.namedWindow("Detected Markers", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Detected Markers", WINDOW_WIDTH, WINDOW_HEIGHT)
    cv2.imshow("Detected Markers", img_with_marker)
    print("Press any key to continue to 3D visualization...")
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    return marker_data, camera_center, camera_rotation

# Main execution
if __name__ == "__main__":
    # Paths to your data
    sfm_file_path = "/home/sha/Documents/mesh/MeshroomCache/StructureFromMotion/366da6468e0b74753b308457693297abf6ed0305/cameras.sfm"
    pcd_path = "/home/sha/Documents/mesh/MeshroomCache/ConvertSfMFormat/ebd642612b8b456855af6468c5b3ed3e2a943192/sfm.ply"
    
    # Process single image and visualize
    marker_data, camera_center, camera_rotation = process_single_image(sfm_file_path, pcd_path)

Marker 1 global position: [-0.32599761 -4.50044677  0.1190539 ]
Point cloud loaded with 70495 points
Added marker 1 to visualization
Displaying 3D visualization...
Press any key to continue to 3D visualization...


In [13]:
from collections import defaultdict


def process_all_markers_from_sfm(sfm_file_path, pcd_path):
    with open(sfm_file_path, "r") as f:
        sfm_data = json.load(f)

    all_markers = defaultdict(list)
    cameras = []

    total_views = len(sfm_data["views"])
    processed_views = 0

    for view in sfm_data["views"]:
        img_path = view["path"]
        viewId = view["viewId"]
        poseId = view["poseId"]

        processed_views += 1
        pose_info = next(
            (
                p["pose"]["transform"]
                for p in sfm_data["poses"]
                if p["poseId"] == poseId
            ),
            None,
        )

        if not pose_info:
            continue

        camera_rotation = np.array(pose_info["rotation"], dtype=float).reshape(3, 3)
        camera_center = np.array(pose_info["center"], dtype=float)

        cameras.append(
            {
                "viewId": viewId,
                "path": img_path,
                "position": camera_center,
                "rotation": camera_rotation,
            }
        )

        img = cv2.imread(img_path)
        if img is None:
            continue

        aruco_dict = cv2.aruco.getPredefinedDictionary(ARUCO_DICT_TYPE)
        parameters = cv2.aruco.DetectorParameters()

        corners, ids, _ = cv2.aruco.detectMarkers(
            img, aruco_dict, parameters=parameters
        )

        if ids is None:
            continue

        rvecs, tvecs, _ = cv2.aruco.estimatePoseSingleMarkers(
            corners, MARKER_LENGTH, CAMERA_MATRIX, DIST_COEFFS
        )

        for i in range(len(ids)):
            marker_id = ids[i][0]
            rvec = rvecs[i]
            tvec = tvecs[i]

            R_marker, _ = cv2.Rodrigues(rvec)

            T_marker_to_cam = np.eye(4)
            T_marker_to_cam[:3, 3] = tvec.flatten()
            T_marker_to_cam[:3, :3] = R_marker

            T_cam_to_global = np.eye(4)
            T_cam_to_global[:3, 3] = camera_center
            T_cam_to_global[:3, :3] = camera_rotation

            T_marker_to_global = T_cam_to_global @ T_marker_to_cam

            position = T_marker_to_global[:3, 3]
            rotation = T_marker_to_global[:3, :3]

            all_markers[marker_id].append(
                {"viewId": viewId, "position": position, "rotation": rotation}
            )

    avg_markers = {}

    for marker_id, positions in all_markers.items():
        sum_pos = np.zeros(3)
        sum_rot = np.zeros((3, 3))
        count = len(positions)

        for pos_info in positions:
            sum_pos += pos_info["position"]
            sum_rot += pos_info["rotation"]

        avg_pos = sum_pos / count
        avg_rot = sum_rot / count

        u, _, vt = np.linalg.svd(avg_rot)
        avg_rot = u @ vt

        avg_markers[marker_id] = {
            "position": avg_pos,
            "rotation": avg_rot,
            "count": count,
        }

    pcd = o3d.io.read_point_cloud(pcd_path)

    geometries = [pcd]

    for cam in cameras:
        camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.05)

        adjusted_rotation = cam["rotation"]
        camera_frame.rotate(adjusted_rotation)
        camera_frame.translate(cam["position"])

        camera_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.01)
        camera_sphere.paint_uniform_color([0, 1, 1])
        camera_sphere.translate(cam["position"])

        geometries.append(camera_frame)
        geometries.append(camera_sphere)

    for marker_id, marker_info in avg_markers.items():
        marker_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)

        marker_frame.translate(marker_info["position"])
        marker_frame.rotate(np.array(marker_info["rotation"]))

        marker_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.02)
        marker_sphere.paint_uniform_color([1, 0, 1])
        marker_sphere.translate(marker_info["position"])

        geometries.append(marker_frame)
        geometries.append(marker_sphere)

    o3d.visualization.draw_geometries(geometries, width=1200, height=800)

    return avg_markers, cameras


sfm_file_path = "/home/sha/Documents/mesh/MeshroomCache/StructureFromMotion/366da6468e0b74753b308457693297abf6ed0305/cameras.sfm"
pcd_path = "/home/sha/Documents/mesh/MeshroomCache/ConvertSfMFormat/ebd642612b8b456855af6468c5b3ed3e2a943192/sfm.ply"

avg_markers, cameras = process_all_markers_from_sfm(sfm_file_path, pcd_path)

output_dir = "/home/sha/Documents/mesh/aruco_results"
os.makedirs(output_dir, exist_ok=True)

serializable_markers = {}
for marker_id, info in avg_markers.items():
    python_marker_id = int(marker_id)
    serializable_markers[python_marker_id] = {
        "position": info["position"].tolist(),
        "rotation": info["rotation"].tolist(),
        "count": info["count"],
    }
with open(os.path.join(output_dir, "averaged_markers.json"), "w") as f:
    json.dump(serializable_markers, f, indent=1)

In [15]:
def load_marker_positions(marker_data_path):
    with open(marker_data_path, 'r') as f:
        marker_data = json.load(f)
    return marker_data

def create_coordinate_system(size=0.1):
    return o3d.geometry.TriangleMesh.create_coordinate_frame(size=size)

def transform_coordinate_system(coord_system, position, rotation_matrix):
    transformed = copy.deepcopy(coord_system)
    transformed.rotate(np.array(rotation_matrix))
    transformed.translate(position)
    return transformed


point_cloud_path = "/home/sha/Documents/mesh/MeshroomCache/ConvertSfMFormat/ebd642612b8b456855af6468c5b3ed3e2a943192/sfm.ply"
marker_data_path = "/home/sha/Documents/mesh/aruco_results/averaged_markers.json"

point_cloud = o3d.io.read_point_cloud(point_cloud_path)

part_centroid = point_cloud.get_center()

marker_data = load_marker_positions(marker_data_path)
coord_system = create_coordinate_system(size=0.1)

part_coord_system = create_coordinate_system(size=0.1)
part_coord_system.translate(part_centroid)

centroid_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.01)
centroid_sphere.paint_uniform_color([1, 1, 0])
centroid_sphere.translate(part_centroid)

marker_coord_systems = []
for marker_id, marker_info in marker_data.items():
    position = np.array(marker_info["position"])
    rotation = np.array(marker_info["rotation"])
    count = marker_info["count"]
    
    transformed_coord = transform_coordinate_system(coord_system, position, rotation)
    marker_coord_systems.append(transformed_coord)
    
    marker_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.01)
    marker_sphere.paint_uniform_color([1, 0, 1])
    marker_sphere.translate(position)
    marker_coord_systems.append(marker_sphere)

geometries = [point_cloud, part_coord_system, centroid_sphere] + marker_coord_systems

o3d.visualization.draw_geometries(
    geometries, 
    zoom=0.7,
)